In [ ]:
# Mount Google Drive and set the data folder
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/Data Hunters'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Load the clean data, group IDs and quality flags (same row order)
import hashlib
import numpy as np
import pandas as pd

# Check that the file is identical to the one in Tuba's README
with open(f'{DATA_DIR}/clean_public_procurement.parquet', 'rb') as f:
    md5 = hashlib.md5(f.read()).hexdigest()
print('MD5:', md5)
print('Matches README:', md5 == '0eab8eb71af6503ede35b2e461a1041d')

df     = pd.read_parquet(f'{DATA_DIR}/clean_public_procurement.parquet')
groups = pd.read_parquet(f'{DATA_DIR}/group_ids.parquet')
flags  = pd.read_parquet(f'{DATA_DIR}/quality_flags.parquet')

print('Row counts:', len(df), len(groups), len(flags))
print('Shape:', df.shape)
df.head()

MD5: 0eab8eb71af6503ede35b2e461a1041d
Matches README: True
Row counts: 365203 365203 365203
Shape: (365203, 22)


,id_type,cae_gpa_annex,iso_country_code,cae_type,main_activity,b_on_behalf,type_of_contract,tal_location_nuts,b_fra_agreement,b_fra_contract,...,gpa_coverage,lots_number,b_eu_funds,top_type,out_of_directives,crit_code,b_electronic_auction,award_value_euro,cpv_div,nuts2
0,3,A1EUN,GR,5,Education,N,S,None,Y,Y,...,2,0,None,OPE,0,M,N,1000000.0,73,None
1,3,A1EUN,GR,5,Education,N,S,GR122,N,N,...,2,0,N,OPE,0,M,N,170333.0,73,GR12
2,3,A1EUN,HU,5,Public Order and Safety,N,S,HU101;HU102,Y,Y,...,2,0,N,OPE,0,L,N,400000.0,79,HU10
3,3,A1EUN,LU,5,Economical anf Financial Affairs,N,S,None,N,N,...,4,2,Y,OPE,0,M,N,55000.0,79,None
4,3,A1EUN,LU,5,Economical anf Financial Affairs,N,S,None,N,N,...,4,2,Y,OPE,0,M,N,55000.0,79,None


In [ ]:
# Column overview: type, number of unique values, share of missing values
overview = pd.DataFrame({
    'dtype':       df.dtypes.astype(str),
    'n_unique':    df.nunique(),
    'pct_missing': (df.isna().mean() * 100).round(1),
    'example':     df.iloc[1],
})
print(overview.to_string())

                        dtype  n_unique  pct_missing    example
id_type                 uint8         3          0.0          3
cae_gpa_annex          object         9          0.0      A1EUN
iso_country_code       object        33          0.0         GR
cae_type               object        10          0.0          5
main_activity          object       262          0.0  Education
b_on_behalf            object         2         10.3          N
type_of_contract       object         3          0.0          S
tal_location_nuts      object      3420         33.7      GR122
b_fra_agreement        object         2          0.0          N
b_fra_contract         object         2          0.0          N
cpv                     int64      5499          0.0   73000000
b_gpa                  object         2         17.2          N
gpa_coverage            uint8         5          0.0          2
lots_number             int64       199          0.0          0
b_eu_funds             object         2 

In [ ]:
# Feature groups
# CPV group = first 3 digits of the CPV code (zfill restores the leading zero, e.g. 03000000)
df['cpv_group'] = df['cpv'].astype(str).str.zfill(8).str[:3]

NUM_COLS  = ['lots_number']
LOW_COLS  = ['id_type', 'cae_gpa_annex', 'iso_country_code', 'cae_type', 'b_on_behalf',
             'type_of_contract', 'b_fra_agreement', 'b_fra_contract', 'b_gpa', 'gpa_coverage',
             'b_eu_funds', 'top_type', 'out_of_directives', 'crit_code', 'b_electronic_auction',
             'cpv_div']                                   # few categories -> one-hot
HIGH_COLS = ['main_activity', 'nuts2', 'cpv_group']       # many categories -> target encoding
FEATURES  = NUM_COLS + LOW_COLS + HIGH_COLS
# Not used: tal_location_nuts (replaced by nuts2), cpv (replaced by cpv_div and cpv_group)

# Treat code-like columns as categories (strings); keep missing values as NaN
for c in LOW_COLS + HIGH_COLS:
    df[c] = df[c].astype(str).where(df[c].notna(), np.nan)

print('Number of features:', len(FEATURES))
print('Numeric:', len(NUM_COLS), '| One-hot:', len(LOW_COLS), '| Target encoding:', len(HIGH_COLS))
print()
print(df[HIGH_COLS].nunique())
print()
print(df[['cpv', 'cpv_div', 'cpv_group']].head(3))

Number of features: 20
Numeric: 1 | One-hot: 16 | Target encoding: 3

main_activity    262
nuts2            528
cpv_group        312
dtype: int64

        cpv cpv_div cpv_group
0  73000000      73       730
1  73000000      73       730
2  79620000      79       796


In [ ]:
# Remove rows below EUR 100, log-transform the target, group-based train/test split
from sklearn.model_selection import GroupShuffleSplit

# Attach notice IDs, row positions and quality flags (all files have the same row order)
df['id_notice_can'] = groups['id_notice_can'].to_numpy()
df['row_id'] = np.arange(len(df))          # position in the clean data, to join titles later
df = pd.concat([df, flags], axis=1)

# Split by notice on ALL rows first; rows below EUR 100 are removed afterwards
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_all, test_all = next(gss.split(df, groups=df['id_notice_can']))

low = df['is_suspect_low'].to_numpy(dtype=bool)
train_idx = train_all[~low[train_all]]
test_idx  = test_all[~low[test_all]]

X = df[FEATURES]
y = np.log1p(df['award_value_euro'].to_numpy())
g = df['id_notice_can'].to_numpy()

X_train, y_train, g_train = X.iloc[train_idx], y[train_idx], g[train_idx]
X_test,  y_test           = X.iloc[test_idx],  y[test_idx]

print('Rows below EUR 100 removed:', low.sum())
print(f'Train: {len(train_idx):,} rows from {len(set(g_train)):,} notices')
print(f'Test:  {len(test_idx):,} rows from {len(set(g[test_idx])):,} notices')
print('Notices in both train and test:', len(set(g_train) & set(g[test_idx])))
print(f'Test share: {len(test_idx) / (len(train_idx) + len(test_idx)):.1%}')

Rows below EUR 100 removed: 17052
Train: 279,299 rows from 88,651 notices
Test:  68,852 rows from 22,160 notices
Notices in both train and test: 0
Test share: 19.8%


In [ ]:
# Metrics and baseline model
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, median_absolute_error

def evaluate(y_true_log, y_pred_log):
    """Metrics on the log scale and, after back-transforming, in euros."""
    y_true_eur, y_pred_eur = np.expm1(y_true_log), np.expm1(y_pred_log)
    return {
        'RMSE (log)':      np.sqrt(mean_squared_error(y_true_log, y_pred_log)),
        'MAE (log)':       mean_absolute_error(y_true_log, y_pred_log),
        'R2 (log)':        r2_score(y_true_log, y_pred_log),
        'MAE (EUR)':       mean_absolute_error(y_true_eur, y_pred_eur),
        'Median AE (EUR)': median_absolute_error(y_true_eur, y_pred_eur),
    }

RESULTS = {}

# Baseline: median log value per CPV division, learned on the training set only
train_medians  = pd.Series(y_train).groupby(X_train['cpv_div'].to_numpy()).median()
overall_median = np.median(y_train)
pred_baseline  = X_test['cpv_div'].map(train_medians).fillna(overall_median).to_numpy()

RESULTS['Baseline'] = evaluate(y_test, pred_baseline)
print(pd.DataFrame(RESULTS).T.round(3).to_string())

          RMSE (log)  MAE (log)  R2 (log)    MAE (EUR)  Median AE (EUR)
Baseline       2.256      1.797     0.271  1634694.142         58073.47


In [ ]:
# Shared preprocessing pipeline
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, TargetEncoder, FunctionTransformer

def make_preprocessor(scale=True, high_cols=None):
    """scale=True for linear models (Ridge); tree models do not need scaling.
    high_cols: columns for target encoding (default: HIGH_COLS)."""
    high_cols = HIGH_COLS if high_cols is None else high_cols

    num_steps = [('impute', SimpleImputer(strategy='median'))]
    if scale:
        num_steps += [('log', FunctionTransformer(np.log1p, feature_names_out='one-to-one')),
                      ('scale', StandardScaler())]

    low_steps = [('impute', SimpleImputer(strategy='constant', fill_value='missing')),
                 ('onehot', OneHotEncoder(handle_unknown='infrequent_if_exist', min_frequency=20))]

    high_steps = [('impute', SimpleImputer(strategy='constant', fill_value='missing')),
                  ('target', TargetEncoder(target_type='continuous', random_state=42))]
    if scale:
        high_steps += [('scale', StandardScaler())]

    return ColumnTransformer([
        ('num',  Pipeline(num_steps),  NUM_COLS),
        ('low',  Pipeline(low_steps),  LOW_COLS),
        ('high', Pipeline(high_steps), high_cols),
    ])

# Quick check: fit on the training data and look at the result
pre = make_preprocessor(scale=True)
X_train_t = pre.fit_transform(X_train, y_train)
names = pre.get_feature_names_out()
print('Shape after preprocessing:', X_train_t.shape)
print('One-hot columns:', sum(n.startswith('low__') for n in names))
print('Examples:', list(names[:3]), '...', list(names[-3:]))

Shape after preprocessing: (279299, 142)
One-hot columns: 138
Examples: ['num__lots_number', 'low__id_type_18', 'low__id_type_3'] ... ['high__main_activity', 'high__nuts2', 'high__cpv_group']


In [ ]:
# Model 1: Ridge regression, alpha tuned with 5-fold GroupKFold CV
import time
from sklearn.linear_model import Ridge
from sklearn.model_selection import GroupKFold, GridSearchCV

cv = GroupKFold(n_splits=5, shuffle=True, random_state=42)   # folds split by notice
CV_RESULTS = {}

ridge = Pipeline([('pre', make_preprocessor(scale=True)), ('model', Ridge())])
search = GridSearchCV(ridge, {'model__alpha': [0.1, 1, 10, 100]},
                      cv=cv, scoring='neg_root_mean_squared_error')

start = time.time()
search.fit(X_train, y_train, groups=g_train)
print(f'Time: {time.time() - start:.0f}s\n')

for a, m, s in zip(search.cv_results_['param_model__alpha'],
                   search.cv_results_['mean_test_score'],
                   search.cv_results_['std_test_score']):
    print(f'alpha = {a:>5}: CV RMSE = {-m:.3f} ± {s:.3f}')
print('Best alpha:', search.best_params_['model__alpha'])

best_ridge = search.best_estimator_            # refit on the full training set with the best alpha
CV_RESULTS['Ridge'] = -search.best_score_
RESULTS['Ridge'] = evaluate(y_test, best_ridge.predict(X_test))
print()
print(pd.DataFrame(RESULTS).T.round(3).to_string())

Time: 150s

alpha =   0.1: CV RMSE = 1.902 ± 0.025
alpha =   1.0: CV RMSE = 1.902 ± 0.025
alpha =  10.0: CV RMSE = 1.902 ± 0.025
alpha = 100.0: CV RMSE = 1.902 ± 0.025
Best alpha: 10

          RMSE (log)  MAE (log)  R2 (log)    MAE (EUR)  Median AE (EUR)
Baseline       2.256      1.797     0.271  1634694.142        58073.470
Ridge          1.887      1.471     0.490  1574617.962        39190.368


In [ ]:
# Model 2: Random Forest
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupShuffleSplit, cross_val_score

# Step 1: tune on a 60k-row subsample (split by notice) with 3-fold CV, to save time
sub, _ = next(GroupShuffleSplit(n_splits=1, train_size=60_000 / len(X_train), random_state=42)
              .split(X_train, groups=g_train))
X_sub, y_sub, g_sub = X_train.iloc[sub], y_train[sub], g_train[sub]
cv3 = GroupKFold(n_splits=3, shuffle=True, random_state=42)

rf = Pipeline([('pre', make_preprocessor(scale=False)),
               ('model', RandomForestRegressor(n_estimators=150, max_samples=0.3,
                                               n_jobs=-1, random_state=42))])
grid = {'model__max_features': [0.2, 0.4], 'model__min_samples_leaf': [3, 10]}

start = time.time()
search = GridSearchCV(rf, grid, cv=cv3, scoring='neg_root_mean_squared_error')
search.fit(X_sub, y_sub, groups=g_sub)
print(f'Tuning time: {time.time() - start:.0f}s')
for p, m in zip(search.cv_results_['params'], search.cv_results_['mean_test_score']):
    print(f'  {p}: RMSE = {-m:.3f}')
print('Best:', search.best_params_)

# Step 2: 5-fold CV on the full training set with the best settings, then fit and test
best_rf = search.best_estimator_
start = time.time()
scores = cross_val_score(best_rf, X_train, y_train, groups=g_train, cv=cv,
                         scoring='neg_root_mean_squared_error')
print(f'\nCV time: {time.time() - start:.0f}s')
print(f'CV RMSE = {-scores.mean():.3f} ± {scores.std():.3f}')

best_rf.fit(X_train, y_train)
CV_RESULTS['Random Forest'] = -scores.mean()
RESULTS['Random Forest'] = evaluate(y_test, best_rf.predict(X_test))
print()
print(pd.DataFrame(RESULTS).T.round(3).to_string())

Tuning time: 513s
  {'model__max_features': 0.2, 'model__min_samples_leaf': 3}: RMSE = 1.920
  {'model__max_features': 0.2, 'model__min_samples_leaf': 10}: RMSE = 1.911
  {'model__max_features': 0.4, 'model__min_samples_leaf': 3}: RMSE = 1.947
  {'model__max_features': 0.4, 'model__min_samples_leaf': 10}: RMSE = 1.938
Best: {'model__max_features': 0.2, 'model__min_samples_leaf': 10}

CV time: 1230s
CV RMSE = 1.748 ± 0.035

               RMSE (log)  MAE (log)  R2 (log)    MAE (EUR)  Median AE (EUR)
Baseline            2.256      1.797     0.271  1634694.142        58073.470
Ridge               1.887      1.471     0.490  1574617.962        39190.368
Random Forest       1.706      1.304     0.583  1525160.402        34955.386


In [ ]:
# Save progress to Drive (so Random Forest does not have to be retrained next session)
import os, joblib
SAVE_DIR = '/content/drive/MyDrive/Data Hunters/modeling'
os.makedirs(SAVE_DIR, exist_ok=True)

joblib.dump(best_ridge, f'{SAVE_DIR}/model_ridge.joblib')
joblib.dump(best_rf,    f'{SAVE_DIR}/model_random_forest.joblib')
pd.DataFrame(RESULTS).T.to_csv(f'{SAVE_DIR}/results_test.csv')
pd.Series(CV_RESULTS, name='CV RMSE (log)').to_csv(f'{SAVE_DIR}/results_cv.csv')
print(os.listdir(SAVE_DIR))

['model_ridge.joblib', 'model_random_forest.joblib', 'results_test.csv', 'results_cv.csv']


In [ ]:
# Model 3: LightGBM (gradient boosting)
from lightgbm import LGBMRegressor
from sklearn.model_selection import RandomizedSearchCV

lgbm = Pipeline([('pre', make_preprocessor(scale=False)),
                 ('model', LGBMRegressor(subsample_freq=1, n_jobs=-1, random_state=42, verbose=-1))])
param_dist = {
    'model__n_estimators':      [300, 600, 1000],
    'model__learning_rate':     [0.03, 0.05, 0.1],
    'model__num_leaves':        [31, 63, 127],
    'model__min_child_samples': [20, 50, 100],
    'model__subsample':         [0.8, 1.0],
    'model__colsample_bytree':  [0.6, 0.8, 1.0],
}

# Step 1: tune on the same 60k-row subsample with 3-fold CV (10 random combinations)
start = time.time()
search = RandomizedSearchCV(lgbm, param_dist, n_iter=10, cv=cv3,
                            scoring='neg_root_mean_squared_error', random_state=42)
search.fit(X_sub, y_sub, groups=g_sub)
print(f'Tuning time: {time.time() - start:.0f}s')
print(f'Best tuning RMSE: {-search.best_score_:.3f}')
print('Best:', search.best_params_)

# Step 2: 5-fold CV on the full training set with the best settings, then fit and test
best_lgbm = search.best_estimator_
start = time.time()
scores = cross_val_score(best_lgbm, X_train, y_train, groups=g_train, cv=cv,
                         scoring='neg_root_mean_squared_error')
print(f'\nCV time: {time.time() - start:.0f}s')
print(f'CV RMSE = {-scores.mean():.3f} ± {scores.std():.3f}')

best_lgbm.fit(X_train, y_train)
CV_RESULTS['LightGBM'] = -scores.mean()
RESULTS['LightGBM'] = evaluate(y_test, best_lgbm.predict(X_test))
print()
print(pd.DataFrame(RESULTS).T.round(3).to_string())

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/v

Tuning time: 283s
Best tuning RMSE: 1.903
Best: {'model__subsample': 1.0, 'model__num_leaves': 31, 'model__n_estimators': 1000, 'model__min_child_samples': 50, 'model__learning_rate': 0.03, 'model__colsample_bytree': 0.8}


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(



CV time: 131s
CV RMSE = 1.754 ± 0.034


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(



               RMSE (log)  MAE (log)  R2 (log)    MAE (EUR)  Median AE (EUR)
Baseline            2.256      1.797     0.271  1634694.142        58073.470
Ridge               1.887      1.471     0.490  1574617.962        39190.368
Random Forest       1.706      1.304     0.583  1525160.402        34955.386
LightGBM            1.704      1.303     0.584  1510492.104        34742.496


In [ ]:
# Final results table and save the LightGBM model
import warnings
warnings.filterwarnings('ignore', message='X does not have valid feature names')

# Baseline CV score on the same 5 folds, for a complete comparison
base_scores = []
for tr, va in cv.split(X_train, y_train, groups=g_train):
    med  = pd.Series(y_train[tr]).groupby(X_train['cpv_div'].iloc[tr].to_numpy()).median()
    pred = X_train['cpv_div'].iloc[va].map(med).fillna(np.median(y_train[tr])).to_numpy()
    base_scores.append(np.sqrt(mean_squared_error(y_train[va], pred)))
CV_RESULTS['Baseline'] = np.mean(base_scores)

final = pd.DataFrame(RESULTS).T
final.insert(0, 'CV RMSE (log)', pd.Series(CV_RESULTS))
final.to_csv(f'{SAVE_DIR}/results_final.csv')

BEST_NAME, best_model = 'LightGBM', best_lgbm
joblib.dump(best_lgbm, f'{SAVE_DIR}/model_lightgbm.joblib')

print(final.round(3).to_string())
print('\nSelected model:', BEST_NAME)
print(os.listdir(SAVE_DIR))

               CV RMSE (log)  RMSE (log)  MAE (log)  R2 (log)    MAE (EUR)  Median AE (EUR)
Baseline               2.235       2.256      1.797     0.271  1634694.142        58073.470
Ridge                  1.902       1.887      1.471     0.490  1574617.962        39190.368
Random Forest          1.748       1.706      1.304     0.583  1525160.402        34955.386
LightGBM               1.754       1.704      1.303     0.584  1510492.104        34742.496

Selected model: LightGBM
['model_ridge.joblib', 'model_random_forest.joblib', 'results_test.csv', 'results_cv.csv', 'results_final.csv', 'model_lightgbm.joblib']


In [ ]:
from sklearn.base import clone
# Sensitivity check: what if the rows below EUR 100 had been kept?
X_train_all, y_train_all = X.iloc[train_all], y[train_all]   # same notices, rows below EUR 100 included
X_test_all,  y_test_all  = X.iloc[test_all],  y[test_all]

start = time.time()
model_with_low = clone(best_model).fit(X_train_all, y_train_all)
print(f'Training time: {time.time() - start:.0f}s\n')

sens = pd.DataFrame({
    '1. Main model (trained and tested without <EUR 100)': evaluate(y_test, best_model.predict(X_test)),
    '2. Trained with <EUR 100, tested without':            evaluate(y_test, model_with_low.predict(X_test)),
    '3. Trained and tested with <EUR 100':                 evaluate(y_test_all, model_with_low.predict(X_test_all)),
}).T
sens.to_csv(f'{SAVE_DIR}/sensitivity_low_values.csv')
print(sens.round(3).to_string())

Training time: 28s

                                                     RMSE (log)  MAE (log)  R2 (log)    MAE (EUR)  Median AE (EUR)
1. Main model (trained and tested without <EUR 100)       1.704      1.303     0.584  1510492.104        34742.496
2. Trained with <EUR 100, tested without                  1.736      1.320     0.568  1524048.495        34094.006
3. Trained and tested with <EUR 100                       1.990      1.447     0.569  1452468.729        30091.391


Note: the shared Data Hunters folder was read-only for me at this point, so the outputs were saved to my own folder in the next cell.

In [ ]:
# Outputs for the analysis step: out-of-fold predictions and residuals for every row
from sklearn.model_selection import cross_val_predict
from sklearn.base import clone

start = time.time()
oof = cross_val_predict(clone(best_model), X_train, y_train, groups=g_train, cv=cv)
pred_test = best_model.predict(X_test)
print(f'Out-of-fold predictions: {time.time() - start:.0f}s')
print('Out-of-fold metrics (train):', {k: round(v, 3) for k, v in evaluate(y_train, oof).items() if 'log' in k})

def make_table(idx, y_log, p_log, split):
    d = df.iloc[idx]
    return pd.DataFrame({
        'row_id':            d['row_id'].to_numpy(),        # row position in clean_public_procurement.parquet
        'id_notice_can':     d['id_notice_can'].to_numpy(),
        'split':             split,
        'actual_eur':        np.expm1(y_log),
        'predicted_eur':     np.expm1(p_log),
        'actual_log':        y_log,
        'predicted_log':     p_log,
        'residual_log':      y_log - p_log,                 # > 0: more expensive than predicted
        'cpv_div':           d['cpv_div'].to_numpy(),
        'iso_country_code':  d['iso_country_code'].to_numpy(),
        'is_repeated_value': d['is_repeated_value'].to_numpy(),
        'is_suspect_high':   d['is_suspect_high'].to_numpy(),
    })

res = pd.concat([make_table(train_idx, y_train, oof, 'train (out-of-fold)'),
                 make_table(test_idx,  y_test,  pred_test, 'test')], ignore_index=True)

# Save to the shared Data Hunters folder so that Irem can use them
OUT_DIR = DATA_DIR
res.to_parquet(f'{OUT_DIR}/predictions_residuals.parquet', index=False)
joblib.dump(best_model, f'{OUT_DIR}/best_model.joblib')
X_test.assign(y_log=y_test, row_id=df['row_id'].iloc[test_idx].to_numpy()) \
      .to_parquet(f'{OUT_DIR}/test_set.parquet', index=False)
final.to_csv(f'{OUT_DIR}/model_results.csv')

# Sanity check: do the records Tuba marked as suspicious get large residuals?
sus = res.loc[res['is_suspect_high'], 'residual_log']
print(f'\nRows saved: {len(res):,}')
print(f'Tuba\'s suspect-high records: {len(sus)}, median residual = {sus.median():.2f}')
print(f'99th percentile of all residuals: {res["residual_log"].quantile(0.99):.2f}')
res.sort_values('residual_log', ascending=False).head(10)

Out-of-fold predictions: 143s
Out-of-fold metrics (train): {'RMSE (log)': np.float64(1.754), 'MAE (log)': 1.343, 'R2 (log)': 0.555}


OSError: [Errno 30] Read-only file system: '/content/drive/MyDrive/Data Hunters/predictions_residuals.parquet'

In [ ]:
# Save outputs to my own folder (the shared Data Hunters folder is read-only for me)
OUT_DIR = SAVE_DIR
res.to_parquet(f'{OUT_DIR}/predictions_residuals.parquet', index=False)
joblib.dump(best_model, f'{OUT_DIR}/best_model.joblib')
X_test.assign(y_log=y_test, row_id=df['row_id'].iloc[test_idx].to_numpy()) \
      .to_parquet(f'{OUT_DIR}/test_set.parquet', index=False)
final.to_csv(f'{OUT_DIR}/model_results.csv')

# Sanity check: do the records Tuba marked as suspicious get large residuals?
sus = res.loc[res['is_suspect_high'], 'residual_log']
print(f'Rows saved: {len(res):,}')
print(f'Tuba\'s suspect-high records: {len(sus)}, median residual = {sus.median():.2f}')
print(f'99th percentile of all residuals: {res["residual_log"].quantile(0.99):.2f}')
print(os.listdir(OUT_DIR))
res.sort_values('residual_log', ascending=False).head(10)

Rows saved: 348,151
Tuba's suspect-high records: 10, median residual = 4.51
99th percentile of all residuals: 4.12
['model_ridge.joblib', 'model_random_forest.joblib', 'results_test.csv', 'results_cv.csv', 'results_final.csv', 'model_lightgbm.joblib', 'sensitivity_low_values.csv', 'predictions_residuals.parquet', 'best_model.joblib', 'test_set.parquet', 'model_results.csv']


,row_id,id_notice_can,split,actual_eur,predicted_eur,actual_log,predicted_log,residual_log,cpv_div,iso_country_code,is_repeated_value,is_suspect_high
257360,337413,2015431326,train (out-of-fold),8.933900e+09,219369.961825,22.913119,12.298519,10.614599,60,FR,False,False
343808,337415,2015431329,test,1.081500e+10,450630.026758,23.104200,13.018404,10.085796,60,FR,False,False
263450,345061,2015439888,train (out-of-fold),1.270834e+10,546501.458918,23.265524,13.211294,10.054230,85,UK,False,True
75423,98711,2015120382,train (out-of-fold),1.312147e+10,630330.215789,23.297516,13.354001,9.943515,85,UK,False,True
338432,305708,2015393914,test,3.209144e+09,256123.780156,21.889270,12.453420,9.435850,33,IT,False,True
117255,153841,2015195938,train (out-of-fold),1.294185e+09,129268.890123,20.981147,11.769658,9.211489,18,PL,False,True
83285,108671,2015136126,train (out-of-fold),2.215579e+07,2852.749283,16.913609,7.956389,8.957220,33,PL,False,False
83289,108675,2015136126,train (out-of-fold),2.033699e+07,2852.749283,16.827952,7.956389,8.871563,33,PL,False,False
133817,175697,2015225974,train (out-of-fold),7.095228e+08,104817.809048,20.380103,11.559989,8.820115,85,UK,False,False
322421,222934,2015290619,test,7.980699e+06,1376.917220,15.892537,7.228328,8.664208,33,RO,False,False


In [ ]:
# Error by country and by CPV division (the data is imbalanced: PL ~34%, CPV 33 ~42%)
from google.colab import drive
drive.mount('/content/drive')
import numpy as np, pandas as pd

SAVE_DIR = '/content/drive/MyDrive/Data Hunters/modeling'
res  = pd.read_parquet(f'{SAVE_DIR}/predictions_residuals.parquet')
test = res[res['split'] == 'test'].assign(abs_err=lambda d: d['residual_log'].abs(),
                                          sq_err=lambda d: d['residual_log'] ** 2)

def errors_by(col, min_rows=300):
    t = test.groupby(col).agg(rows=('abs_err', 'size'),
                              rmse_log=('sq_err', lambda s: np.sqrt(s.mean())),
                              mae_log=('abs_err', 'mean'),
                              median_value_eur=('actual_eur', 'median'))
    return t[t['rows'] >= min_rows].sort_values('rows', ascending=False).round(3)

by_country = errors_by('iso_country_code')
by_cpv     = errors_by('cpv_div')
by_country.to_csv(f'{SAVE_DIR}/errors_by_country.csv')
by_cpv.to_csv(f'{SAVE_DIR}/errors_by_cpv.csv')

print('Overall test RMSE (log):', round(np.sqrt(test['sq_err'].mean()), 3))
print('\nBy country:\n', by_country.to_string())
print('\nBy CPV division (top 15):\n', by_cpv.head(15).to_string())

Mounted at /content/drive
Overall test RMSE (log): 1.704

By country:
                    rows  rmse_log  mae_log  median_value_eur
iso_country_code                                            
PL                24631     1.792    1.396         10038.000
FR                10289     1.544    1.181         86751.750
RO                 6230     2.059    1.667         11690.290
IT                 3417     1.652    1.259        186000.000
DE                 3238     1.408    1.058        197365.070
ES                 3022     1.476    1.099        194850.000
CZ                 2612     1.336    0.937         56030.700
BG                 1655     1.846    1.440         25634.780
LV                 1519     1.905    1.502         95685.000
LT                 1468     1.694    1.332         17217.000
UK                 1448     1.611    1.185        478582.745
BE                 1091     1.595    1.108        125053.000
GR                 1055     1.322    1.011         36796.880
HR            

In [ ]:
# Experiment: which low-value filter for training? (CV inside the training notices, test set untouched)
import time, joblib
from sklearn.base import clone
from sklearn.model_selection import GroupKFold

SAVE_DIR = '/content/drive/MyDrive/Data Hunters/modeling'
lgbm_template = joblib.load(f'{SAVE_DIR}/model_lightgbm.joblib')   # tuned LightGBM pipeline
cv = GroupKFold(n_splits=5, shuffle=True, random_state=42)

# All training notices, including the rows below EUR 100
X_tr, y_tr, g_tr = X.iloc[train_all], y[train_all], g[train_all]
value = df['award_value_euro'].to_numpy()[train_all]

scenarios = {'No filter': 0, 'Remove < EUR 1': 1, 'Remove < EUR 100': 100}
rows = []
start = time.time()
for fold, (tr, va) in enumerate(cv.split(X_tr, y_tr, groups=g_tr), 1):
    va = va[value[va] >= 100]                       # every scenario is validated on the same lots (>= EUR 100)
    for name, limit in scenarios.items():
        tr_k = tr[value[tr] >= limit]               # only the training rows change
        m = clone(lgbm_template).fit(X_tr.iloc[tr_k], y_tr[tr_k])
        r = evaluate(y_tr[va], m.predict(X_tr.iloc[va]))
        rows.append({'fold': fold, 'scenario': name, 'train_rows': len(tr_k),
                     'RMSE (log)': r['RMSE (log)'], 'Median AE (EUR)': r['Median AE (EUR)']})
    print(f'Fold {fold} done ({time.time() - start:.0f}s)')

exp = pd.DataFrame(rows)
summary = exp.groupby('scenario').agg(train_rows=('train_rows', 'mean'),
                                      rmse_mean=('RMSE (log)', 'mean'), rmse_std=('RMSE (log)', 'std'),
                                      median_ae_eur=('Median AE (EUR)', 'mean')).loc[list(scenarios)]
summary.to_csv(f'{SAVE_DIR}/experiment_low_value_filter.csv')
print()
print(summary.round(3).to_string())

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Fold 1 done (76s)


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Fold 2 done (151s)


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Fold 3 done (227s)


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Fold 4 done (313s)


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Fold 5 done (389s)

                  train_rows  rmse_mean  rmse_std  median_ae_eur
scenario                                                        
No filter           234269.6      1.755     0.042      30581.447
Remove < EUR 1      233736.0      1.750     0.032      30099.474
Remove < EUR 100    223439.2      1.733     0.042      30205.590


In [ ]:
# Paired comparison: is the EUR 100 filter better in every fold?
per_fold = exp.pivot(index='fold', columns='scenario', values='RMSE (log)')[list(scenarios)]
per_fold['100 vs no filter'] = per_fold['Remove < EUR 100'] - per_fold['No filter']
per_fold['100 vs 1']         = per_fold['Remove < EUR 100'] - per_fold['Remove < EUR 1']
print(per_fold.round(3).to_string())
print('\nFolds where EUR 100 is better than no filter:', (per_fold['100 vs no filter'] < 0).sum(), 'of 5')
print('Folds where EUR 100 is better than EUR 1:    ', (per_fold['100 vs 1'] < 0).sum(), 'of 5')

scenario  No filter  Remove < EUR 1  Remove < EUR 100  100 vs no filter  100 vs 1
fold                                                                             
1             1.734           1.732             1.694            -0.040    -0.037
2             1.768           1.767             1.776             0.009     0.009
3             1.821           1.796             1.776            -0.046    -0.021
4             1.740           1.738             1.728            -0.012    -0.010
5             1.712           1.716             1.689            -0.023    -0.027

Folds where EUR 100 is better than no filter: 4 of 5
Folds where EUR 100 is better than EUR 1:     4 of 5


In [ ]:
# Experiment: add the full 8-digit CPV code as a target-encoded feature (CV on the training set)
from lightgbm import LGBMRegressor

X_cpv = X_train.assign(cpv=df['cpv'].iloc[train_idx].astype(str).str.zfill(8).to_numpy())
lgbm_params = lgbm_template.named_steps['model'].get_params()      # same tuned LightGBM settings

def lgbm_with(high_cols):
    return Pipeline([('pre', make_preprocessor(scale=False, high_cols=high_cols)),
                     ('model', LGBMRegressor(**lgbm_params))])

variants = {'Current features': HIGH_COLS, '+ full CPV code': HIGH_COLS + ['cpv']}
is33 = (X_train['cpv_div'] == '33').to_numpy()

rows = []
start = time.time()
for fold, (tr, va) in enumerate(cv.split(X_cpv, y_train, groups=g_train), 1):
    for name, hc in variants.items():
        m = lgbm_with(hc).fit(X_cpv.iloc[tr], y_train[tr])
        err = y_train[va] - m.predict(X_cpv.iloc[va])
        rows.append({'fold': fold, 'variant': name,
                     'RMSE (log)':  np.sqrt(np.mean(err ** 2)),
                     'RMSE CPV 33': np.sqrt(np.mean(err[is33[va]] ** 2)),
                     'RMSE other':  np.sqrt(np.mean(err[~is33[va]] ** 2))})
    print(f'Fold {fold} done ({time.time() - start:.0f}s)')

exp_cpv = pd.DataFrame(rows)
summary_cpv = exp_cpv.groupby('variant')[['RMSE (log)', 'RMSE CPV 33', 'RMSE other']].mean().loc[list(variants)]
summary_cpv.to_csv(f'{SAVE_DIR}/experiment_full_cpv.csv')
print()
print(summary_cpv.round(3).to_string())

diff = exp_cpv.pivot(index='fold', columns='variant', values='RMSE (log)')
print('\nFolds where the full CPV code helps:', (diff['+ full CPV code'] < diff['Current features']).sum(), 'of 5')

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Fold 1 done (72s)


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Fold 2 done (132s)


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Fold 3 done (184s)


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Fold 4 done (233s)


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Fold 5 done (291s)

                  RMSE (log)  RMSE CPV 33  RMSE other
variant                                              
Current features       1.754        2.007       1.550
+ full CPV code        1.759        2.008       1.559

Folds where the full CPV code helps: 2 of 5


In [ ]:
# Diagnostic: how much of the variation is between lots of the same notice?
tr_df = df.iloc[train_idx]
ylog  = np.log1p(tr_df['award_value_euro'])

# 1) Share of the target variance that lies WITHIN notices
within = (ylog - ylog.groupby(tr_df['id_notice_can']).transform('mean')) ** 2
print(f'Share of variance within notices: {within.mean() / ylog.var(ddof=0):.1%}')

# 2) Do lots of the same notice have different CPV codes?
multi = tr_df.groupby('id_notice_can')['row_id'].transform('size') > 1
print(f'Rows in multi-lot notices: {multi.mean():.1%}')
cpv_n = tr_df[multi].groupby('id_notice_can')['cpv'].nunique()
print(f'Multi-lot notices whose lots have different CPV codes: {(cpv_n > 1).mean():.1%}')

# 3) Titles: how often do they exist, and do they differ between lots? (same alignment as in Tuba's EDA)
raw = pd.read_parquet(f'{DATA_DIR}/raw_public_procurement.parquet', columns=['award_value_euro', 'title'])
raw = raw[raw['award_value_euro'] > 0].reset_index(drop=True)
assert len(raw) == len(groups), 'Raw data is not aligned with the clean data'
tr_df = tr_df.assign(title=raw['title'].iloc[tr_df['row_id']].to_numpy())
print(f'\nRows with a title: {tr_df["title"].notna().mean():.1%}')
title_n = tr_df[multi].groupby('id_notice_can')['title'].nunique()
print(f'Multi-lot notices whose lots have different titles: {(title_n > 1).mean():.1%}')
print('\nExample titles:')
print(tr_df['title'].dropna().sample(8, random_state=1).str[:90].to_string())

Share of variance within notices: 17.7%
Rows in multi-lot notices: 77.0%
Multi-lot notices whose lots have different CPV codes: 0.0%

Rows with a title: 59.5%
Multi-lot notices whose lots have different titles: 70.2%

Example titles:
119791                    UV laser pro konfokální mikroskop
95319                                  Обособена позиция 73
283067    Sozialmedizinisches Zentrum Baumgartner Höhe –...
71018               Grupa 53. Podkłady i pozycjonery żelowe
230741                                   Produkty lecznicze
351875                                     Campus Heide-Süd
66293     Dostawa leków i opatrunków dla Powiatowego Cen...
82982            Trastuzumab 600 mg/5 ml sol inj fr 5 ml sc


In [ ]:
# Experiment: add the tender title as TF-IDF features (character n-grams, works across languages)
import warnings
warnings.filterwarnings('ignore', message='X does not have valid feature names')
from sklearn.feature_extraction.text import TfidfVectorizer

X_title = X_train.assign(title=raw['title'].iloc[df['row_id'].iloc[train_idx]].fillna('').to_numpy())

def lgbm_with_title(max_features=5000):
    pre = ColumnTransformer([
        ('base', make_preprocessor(scale=False), FEATURES),
        ('text', TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), min_df=20,
                                 max_features=max_features, sublinear_tf=True), 'title'),
    ])
    return Pipeline([('pre', pre), ('model', LGBMRegressor(**lgbm_params))])

rows = []
start = time.time()
for fold, (tr, va) in enumerate(cv.split(X_title, y_train, groups=g_train), 1):
    m = lgbm_with_title().fit(X_title.iloc[tr], y_train[tr])
    err = y_train[va] - m.predict(X_title.iloc[va])
    rows.append({'fold': fold, 'variant': '+ title (TF-IDF)',
                 'RMSE (log)':  np.sqrt(np.mean(err ** 2)),
                 'RMSE CPV 33': np.sqrt(np.mean(err[is33[va]] ** 2)),
                 'RMSE other':  np.sqrt(np.mean(err[~is33[va]] ** 2))})
    print(f'Fold {fold} done ({time.time() - start:.0f}s)')

# Compare with the current model on the same folds (results from the previous experiment)
exp_title = pd.concat([exp_cpv[exp_cpv['variant'] == 'Current features'], pd.DataFrame(rows)])
summary_title = exp_title.groupby('variant')[['RMSE (log)', 'RMSE CPV 33', 'RMSE other']].mean()
summary_title.to_csv(f'{SAVE_DIR}/experiment_title_tfidf.csv')
print()
print(summary_title.round(3).to_string())

d = exp_title.pivot(index='fold', columns='variant', values='RMSE (log)')
print('\nFolds where the title helps:', (d['+ title (TF-IDF)'] < d['Current features']).sum(), 'of 5')

Fold 1 done (567s)
Fold 2 done (1119s)
Fold 3 done (1681s)
Fold 4 done (2251s)
Fold 5 done (2808s)

                  RMSE (log)  RMSE CPV 33  RMSE other
variant                                              
+ title (TF-IDF)       1.735        1.963       1.554
Current features       1.754        2.007       1.550

Folds where the title helps: 5 of 5
